# Week 1 – Agricultural Data Acquisition and Cleaning
## Crop Production in India (Kaggle)

**Tools:** Google Colab, Python, Pandas and DuckDB SQL  
**Dataset source:** [Crop Production in India – Kaggle](https://www.kaggle.com/datasets/abhinand05/crop-production-in-india)

### Project objective
Acquire a public agricultural dataset, inspect its structure and quality, load it into a SQL staging table, identify and handle data-quality issues, create a cleaned analytical table, and perform preliminary transformations.

### Important execution notes
1. Download the CSV from Kaggle and upload it when prompted below.
2. This notebook is designed to identify the seven expected fields: `State_Name`, `District_Name`, `Crop_Year`, `Season`, `Crop`, `Area`, and `Production`. It also accepts common capitalization/spacing variations.
3. Review all data-quality outputs before accepting the cleaned table. Rows excluded from the clean table are preserved in a review table.
4. Do not invent or copy example statistics into your report. Use the actual outputs generated by running this notebook.

## 1. Install and import required libraries

In [ ]:
!pip -q install duckdb

import pandas as pd
import duckdb
import re
from pathlib import Path
from google.colab import files
from IPython.display import display

print("Pandas version:", pd.__version__)
print("DuckDB version:", duckdb.__version__)

## 2. Acquire the Kaggle CSV file
Download the dataset from Kaggle using the link above. If Kaggle requires login, sign in and download the CSV ZIP/file manually. Upload the extracted CSV here.

In [ ]:
uploaded = files.upload()

csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("No CSV file uploaded. Please upload the extracted Kaggle CSV file.")

CSV_PATH = csv_files[0]
print("Selected file:", CSV_PATH)

## 3. Load and inspect the raw dataset
The original CSV is loaded without overwriting it. We inspect its dimensions, headers, data types, sample records and missing values.

In [ ]:
raw_df = pd.read_csv(CSV_PATH, low_memory=False)
print("Rows:", raw_df.shape[0])
print("Columns:", raw_df.shape[1])
print("\\nColumn names:")
print(raw_df.columns.tolist())
display(raw_df.head(10))
display(raw_df.dtypes.rename("data_type").to_frame())

## 4. Standardise source column names and validate schema
This maps common header variations to the expected project schema. If a required field is absent, the notebook stops rather than silently using the wrong columns.

In [ ]:
def normalise_header(name):
    return re.sub(r"[^a-z0-9]", "", str(name).strip().lower())

header_lookup = {normalise_header(c): c for c in raw_df.columns}
aliases = {
    "state_name": ["statename", "state"],
    "district_name": ["districtname", "district"],
    "crop_year": ["cropyear", "year"],
    "season": ["season"],
    "crop": ["crop"],
    "area": ["area"],
    "production": ["production"]
}

rename_map = {}
missing_fields = []
for target, candidates in aliases.items():
    found = next((header_lookup[a] for a in candidates if a in header_lookup), None)
    if found is None:
        missing_fields.append(target)
    else:
        rename_map[found] = target

if missing_fields:
    raise ValueError(
        "The uploaded CSV does not contain these required fields: "
        + ", ".join(missing_fields)
        + ". Check that you uploaded the correct Kaggle dataset/version."
    )

df = raw_df.rename(columns=rename_map)[
    ["state_name", "district_name", "crop_year", "season", "crop", "area", "production"]
].copy()

print("Validated project columns:")
print(df.columns.tolist())
display(df.head())

## 5. Create a SQL database and load the raw staging table
DuckDB runs SQL directly inside Colab. The staging table preserves source values as text so that malformed numeric values can be detected before conversion.

In [ ]:
con = duckdb.connect("agriculture_week1.duckdb")

# Preserve source values as strings; convert pandas missing values to SQL NULL.
raw_for_sql = df.astype("string")
con.register("incoming_crop_data", raw_for_sql)
con.execute("DROP TABLE IF EXISTS raw_crop_production")
con.execute("CREATE TABLE raw_crop_production AS SELECT * FROM incoming_crop_data")
con.unregister("incoming_crop_data")

print("Raw SQL table created.")
display(con.execute("SELECT COUNT(*) AS total_raw_records FROM raw_crop_production").df())
display(con.execute("DESCRIBE raw_crop_production").df())
display(con.execute("SELECT * FROM raw_crop_production LIMIT 10").df())

## 6. Data-quality assessment using SQL
The following queries count missing/blank fields, inspect exact duplicate records, identify invalid numeric strings, negative measurements and non-four-digit year values.

In [ ]:
missing_sql = con.execute("""
SELECT
    SUM(CASE WHEN state_name IS NULL OR TRIM(state_name) = '' THEN 1 ELSE 0 END) AS missing_state,
    SUM(CASE WHEN district_name IS NULL OR TRIM(district_name) = '' THEN 1 ELSE 0 END) AS missing_district,
    SUM(CASE WHEN crop_year IS NULL OR TRIM(crop_year) = '' THEN 1 ELSE 0 END) AS missing_crop_year,
    SUM(CASE WHEN season IS NULL OR TRIM(season) = '' THEN 1 ELSE 0 END) AS missing_season,
    SUM(CASE WHEN crop IS NULL OR TRIM(crop) = '' THEN 1 ELSE 0 END) AS missing_crop,
    SUM(CASE WHEN area IS NULL OR TRIM(area) = '' THEN 1 ELSE 0 END) AS missing_area,
    SUM(CASE WHEN production IS NULL OR TRIM(production) = '' THEN 1 ELSE 0 END) AS missing_production
FROM raw_crop_production
""").df()
display(missing_sql)

duplicate_sql = con.execute("""
SELECT
    TRIM(state_name) AS state_name,
    TRIM(district_name) AS district_name,
    TRIM(crop_year) AS crop_year,
    TRIM(season) AS season,
    TRIM(crop) AS crop,
    TRIM(area) AS area,
    TRIM(production) AS production,
    COUNT(*) AS occurrences
FROM raw_crop_production
GROUP BY ALL
HAVING COUNT(*) > 1
ORDER BY occurrences DESC
""").df()
print("Exact duplicate groups:", len(duplicate_sql))
display(duplicate_sql.head(20))

invalid_numeric = con.execute("""
SELECT *
FROM raw_crop_production
WHERE
    (NULLIF(TRIM(area), '') IS NOT NULL
     AND TRY_CAST(TRIM(area) AS DOUBLE) IS NULL)
 OR (NULLIF(TRIM(production), '') IS NOT NULL
     AND TRY_CAST(TRIM(production) AS DOUBLE) IS NULL)
""").df()
print("Rows with non-numeric Area or Production:", len(invalid_numeric))
display(invalid_numeric.head(20))

negative_values = con.execute("""
SELECT *
FROM raw_crop_production
WHERE TRY_CAST(TRIM(area) AS DOUBLE) < 0
   OR TRY_CAST(TRIM(production) AS DOUBLE) < 0
""").df()
print("Rows with negative Area or Production:", len(negative_values))
display(negative_values.head(20))

invalid_years = con.execute("""
SELECT DISTINCT crop_year
FROM raw_crop_production
WHERE NOT REGEXP_MATCHES(TRIM(CAST(crop_year AS VARCHAR)), '^[0-9]{4}$')
ORDER BY crop_year
""").df()
print("Year values not matching four digits:", len(invalid_years))
display(invalid_years.head(20))

## 7. Create a normalised SQL working table
Trim leading/trailing spaces and convert empty strings to NULL. The raw staging table remains unchanged.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE crop_production_normalised AS
SELECT
    ROW_NUMBER() OVER () AS source_row_id,
    NULLIF(TRIM(CAST(state_name AS VARCHAR)), '') AS state_name,
    NULLIF(TRIM(CAST(district_name AS VARCHAR)), '') AS district_name,
    NULLIF(TRIM(CAST(crop_year AS VARCHAR)), '') AS crop_year_text,
    NULLIF(TRIM(CAST(season AS VARCHAR)), '') AS season,
    NULLIF(TRIM(CAST(crop AS VARCHAR)), '') AS crop_name,
    NULLIF(TRIM(CAST(area AS VARCHAR)), '') AS area_text,
    NULLIF(TRIM(CAST(production AS VARCHAR)), '') AS production_text
FROM raw_crop_production
""")
display(con.execute("SELECT * FROM crop_production_normalised LIMIT 10").df())

## 8. Separate records requiring review
Rows with missing required fields, malformed years/numbers, or negative area/production are preserved in a review table with a reason. This avoids silently deleting questionable records.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE crop_production_review AS
SELECT *,
    CONCAT_WS('; ',
        CASE WHEN state_name IS NULL THEN 'Missing state' END,
        CASE WHEN district_name IS NULL THEN 'Missing district' END,
        CASE WHEN crop_year_text IS NULL THEN 'Missing crop year' END,
        CASE WHEN crop_year_text IS NOT NULL
              AND NOT REGEXP_MATCHES(crop_year_text, '^[0-9]{4}$')
             THEN 'Invalid crop year format' END,
        CASE WHEN season IS NULL THEN 'Missing season' END,
        CASE WHEN crop_name IS NULL THEN 'Missing crop' END,
        CASE WHEN area_text IS NULL THEN 'Missing area' END,
        CASE WHEN area_text IS NOT NULL AND TRY_CAST(area_text AS DOUBLE) IS NULL
             THEN 'Area is not numeric' END,
        CASE WHEN production_text IS NULL THEN 'Missing production' END,
        CASE WHEN production_text IS NOT NULL AND TRY_CAST(production_text AS DOUBLE) IS NULL
             THEN 'Production is not numeric' END,
        CASE WHEN TRY_CAST(area_text AS DOUBLE) < 0 THEN 'Negative area' END,
        CASE WHEN TRY_CAST(production_text AS DOUBLE) < 0 THEN 'Negative production' END
    ) AS review_reason
FROM crop_production_normalised
WHERE state_name IS NULL
   OR district_name IS NULL
   OR crop_year_text IS NULL
   OR NOT REGEXP_MATCHES(COALESCE(crop_year_text, ''), '^[0-9]{4}$')
   OR season IS NULL
   OR crop_name IS NULL
   OR area_text IS NULL
   OR TRY_CAST(area_text AS DOUBLE) IS NULL
   OR TRY_CAST(area_text AS DOUBLE) < 0
   OR production_text IS NULL
   OR TRY_CAST(production_text AS DOUBLE) IS NULL
   OR TRY_CAST(production_text AS DOUBLE) < 0
""")
print("Rows requiring review:", con.execute("SELECT COUNT(*) FROM crop_production_review").fetchone()[0])
display(con.execute("SELECT * FROM crop_production_review LIMIT 20").df())

## 9. Create the cleaned analytical table
Only rows that pass required-field, year, numeric and non-negative checks are eligible. Exact duplicate source records are reduced to one representative using `ROW_NUMBER()`. Records sharing crop/location/year but having different measurements are not automatically treated as duplicates. Yield is calculated as production divided by area only when area is greater than zero.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE clean_crop_production AS
WITH eligible AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY
                LOWER(TRIM(state_name)),
                LOWER(TRIM(district_name)),
                crop_year_text,
                LOWER(TRIM(season)),
                LOWER(TRIM(crop_name)),
                area_text,
                production_text
            ORDER BY source_row_id
        ) AS duplicate_rank
    FROM crop_production_normalised
    WHERE state_name IS NOT NULL
      AND district_name IS NOT NULL
      AND crop_year_text IS NOT NULL
      AND REGEXP_MATCHES(crop_year_text, '^[0-9]{4}$')
      AND season IS NOT NULL
      AND crop_name IS NOT NULL
      AND TRY_CAST(area_text AS DOUBLE) IS NOT NULL
      AND TRY_CAST(production_text AS DOUBLE) IS NOT NULL
      AND TRY_CAST(area_text AS DOUBLE) >= 0
      AND TRY_CAST(production_text AS DOUBLE) >= 0
)
SELECT
    source_row_id,
    TRIM(state_name) AS state_name,
    TRIM(district_name) AS district_name,
    CAST(crop_year_text AS INTEGER) AS crop_year,
    TRIM(season) AS season,
    TRIM(crop_name) AS crop_name,
    CAST(area_text AS DOUBLE) AS area,
    CAST(production_text AS DOUBLE) AS production,
    CASE WHEN CAST(area_text AS DOUBLE) > 0
         THEN CAST(production_text AS DOUBLE) / CAST(area_text AS DOUBLE)
         ELSE NULL END AS yield_per_area
FROM eligible
WHERE duplicate_rank = 1
""")

print("Cleaned table created.")
display(con.execute("SELECT * FROM clean_crop_production LIMIT 10").df())

## 10. Post-cleaning validation and reconciliation
Compare raw, review and cleaned counts. Exact duplicates removed from otherwise eligible records are tracked separately; therefore, raw count need not equal review count plus cleaned count.

In [ ]:
counts = con.execute("""
SELECT
    (SELECT COUNT(*) FROM raw_crop_production) AS raw_records,
    (SELECT COUNT(*) FROM crop_production_review) AS review_records,
    (SELECT COUNT(*) FROM clean_crop_production) AS clean_records
""").df()
display(counts)

null_check = con.execute("""
SELECT
    SUM(CASE WHEN state_name IS NULL THEN 1 ELSE 0 END) AS null_state,
    SUM(CASE WHEN district_name IS NULL THEN 1 ELSE 0 END) AS null_district,
    SUM(CASE WHEN crop_year IS NULL THEN 1 ELSE 0 END) AS null_year,
    SUM(CASE WHEN season IS NULL THEN 1 ELSE 0 END) AS null_season,
    SUM(CASE WHEN crop_name IS NULL THEN 1 ELSE 0 END) AS null_crop,
    SUM(CASE WHEN area IS NULL THEN 1 ELSE 0 END) AS null_area,
    SUM(CASE WHEN production IS NULL THEN 1 ELSE 0 END) AS null_production
FROM clean_crop_production
""").df()
display(null_check)

range_check = con.execute("""
SELECT
    MIN(area) AS min_area,
    MAX(area) AS max_area,
    MIN(production) AS min_production,
    MAX(production) AS max_production,
    MIN(yield_per_area) AS min_yield_per_area,
    MAX(yield_per_area) AS max_yield_per_area
FROM clean_crop_production
""").df()
display(range_check)

## 11. Preliminary transformations using SQL
These summaries demonstrate that the cleaned data can support agricultural analysis. Interpret units according to the dataset documentation; do not assume production units without confirming the source metadata.

In [ ]:
print("Top states by total production")
display(con.execute("""
SELECT state_name, ROUND(SUM(production), 2) AS total_production
FROM clean_crop_production
GROUP BY state_name
ORDER BY total_production DESC
LIMIT 15
""").df())

print("Top crops by total production")
display(con.execute("""
SELECT crop_name, ROUND(SUM(production), 2) AS total_production
FROM clean_crop_production
GROUP BY crop_name
ORDER BY total_production DESC
LIMIT 15
""").df())

print("Production by crop year")
display(con.execute("""
SELECT crop_year, ROUND(SUM(production), 2) AS total_production
FROM clean_crop_production
GROUP BY crop_year
ORDER BY crop_year
""").df())

print("Production by season")
display(con.execute("""
SELECT season, ROUND(SUM(production), 2) AS total_production
FROM clean_crop_production
GROUP BY season
ORDER BY total_production DESC
""").df())

print("Average yield proxy by crop (production / area)")
display(con.execute("""
SELECT crop_name, ROUND(AVG(yield_per_area), 4) AS average_yield_per_area
FROM clean_crop_production
WHERE yield_per_area IS NOT NULL
GROUP BY crop_name
ORDER BY average_yield_per_area DESC
LIMIT 15
""").df())

## 12. Create a reusable summary view

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW vw_state_crop_summary AS
SELECT
    state_name,
    crop_name,
    COUNT(*) AS record_count,
    ROUND(SUM(area), 2) AS total_area,
    ROUND(SUM(production), 2) AS total_production,
    ROUND(AVG(yield_per_area), 4) AS average_yield_per_area
FROM clean_crop_production
GROUP BY state_name, crop_name
""")
display(con.execute("SELECT * FROM vw_state_crop_summary ORDER BY total_production DESC LIMIT 20").df())

## 13. Export cleaned data, review records and summary outputs
These files can be attached as supporting evidence with the written report.

In [ ]:
clean_df = con.execute("SELECT * FROM clean_crop_production").df()
review_df = con.execute("SELECT * FROM crop_production_review").df()
summary_df = con.execute("SELECT * FROM vw_state_crop_summary").df()

clean_df.to_csv("clean_crop_production.csv", index=False)
review_df.to_csv("crop_production_review.csv", index=False)
summary_df.to_csv("state_crop_summary.csv", index=False)

# Save the DuckDB database file as well as CSV outputs.
con.close()

print("Created files:")
for filename in [
    "clean_crop_production.csv",
    "crop_production_review.csv",
    "state_crop_summary.csv",
    "agriculture_week1.duckdb"
]:
    print(filename, Path(filename).stat().st_size, "bytes")

## 14. Download project outputs

In [ ]:
for filename in [
    "clean_crop_production.csv",
    "crop_production_review.csv",
    "state_crop_summary.csv",
    "agriculture_week1.duckdb"
]:
    files.download(filename)

## 15. Report-writing notes

After running every cell, record the actual outputs in your DOCX report:

- Original row and column counts.
- Missing-value counts by field.
- Number of exact duplicate groups and duplicate rows.
- Number of records with invalid numeric values, negative values and invalid years.
- Number of rows placed in the review table.
- Number of rows in the final cleaned table.
- Key findings from state-wise, crop-wise, year-wise and season-wise summaries.

### Methodological cautions
- A blank value is not automatically zero. Missing measurements are retained for review rather than filled with zero.
- Exact duplicates are identified using all source fields. Similar crop/location/year records with different measurements are not automatically deleted.
- The yield field is a calculated ratio (`production / area`). Confirm the source units before interpreting it as a standard agronomic yield.
- This notebook uses DuckDB SQL in Google Colab, not a remote MySQL server. It demonstrates SQL data acquisition, cleaning and transformation in a Colab environment.
- Include screenshots of your own executed cells and outputs. Do not report results until the notebook has run successfully on the downloaded Kaggle file.